<a href="https://colab.research.google.com/github/nikitha01reddy/NLP_multilingual_govt_scheme_FAQ/blob/main/irreleventFIX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# INSTALL
!pip install -q openai-whisper transformers sentence-transformers faiss-cpu gTTS pydub soundfile librosa
!apt-get install ffmpeg -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.8 MB/s eta 0:00:00


In [2]:
import whisper
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from gtts import gTTS
from pydub import AudioSegment
from google.colab import files
from IPython.display import Audio


# SPEECH TO TEXT (WHISPER)
class WhisperSTT:

    def __init__(self):
        print("Loading Whisper model...")
        self.model = whisper.load_model("medium")

    def speech_to_text(self, audio_path):
        result = self.model.transcribe(
            audio_path,
            task="translate",
            language="te",
            fp16=False
        )
        return str(result["text"]).strip()


# TRANSLATOR (NLLB)
class Translator:

    def __init__(self):

        print("Loading translation model...")

        model_name = "facebook/nllb-200-distilled-600M"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        self.lang_map = {
            "eng": "eng_Latn",
            "hin": "hin_Deva",
            "tel": "tel_Telu",
            "ben": "ben_Beng",
            "urd": "urd_Arab"
        }

    def translate(self, text, target_lang):

        tgt = self.lang_map.get(target_lang, "eng_Latn")

        inputs = self.tokenizer(text, return_tensors="pt")

        forced_bos_token_id = self.tokenizer.convert_tokens_to_ids(tgt)

        output = self.model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id
        )

        return self.tokenizer.batch_decode(output, skip_special_tokens=True)[0]


# RETRIEVER
class JinaRetriever:

    def __init__(self):

        print("Loading embedding model...")

        self.encoder = SentenceTransformer("intfloat/multilingual-e5-base")

        self.index = None
        self.docs = []

    def build_index(self, documents):

        prefixed_docs = ["passage: " + doc for doc in documents]

        embeddings = self.encoder.encode(
            prefixed_docs,
            convert_to_numpy=True
        )

        print("Embedding shape:", embeddings.shape)

        dimension = embeddings.shape[1]

        self.index = faiss.IndexFlatIP(dimension)

        faiss.normalize_L2(embeddings)

        self.index.add(embeddings)

        self.docs = [{"text": doc} for doc in documents]

    def retrieve(self, query, top_k=3):

        query = "query: " + str(query)

        query_emb = self.encoder.encode(
            [query],
            convert_to_numpy=True
        )

        faiss.normalize_L2(query_emb)

        scores, indices = self.index.search(query_emb, top_k)

        results = []

        for score, idx in zip(scores[0], indices[0]):
            results.append({
                "text": self.docs[idx]["text"],
                "score": float(score)
            })

        return results


# TEXT TO SPEECH
class LocalTTS:

    def synthesize(self, text, lang="en", output="answer.mp3"):

        tts = gTTS(text=text, lang=lang)

        tts.save(output)

        return output


# PIPELINE
class IndicQAPipeline:

    def __init__(self, stt_model, retriever, translator, tts_model, lang_map):

        self.stt = stt_model
        self.retriever = retriever
        self.translator = translator
        self.tts = tts_model
        self.lang_map = lang_map

        # scheme keywords for relevance detection
        self.scheme_keywords = [
            "bharati", "kisan", "ayushman", "rythu", "bima",
            "scheme", "yojana", "farmer", "Ayushman Bharat "
        ]


    def is_relevant(self, query, score):
        query_lower = query.lower()
        keyword_match = any(k in query_lower for k in self.scheme_keywords)

        if keyword_match:
            # Queries that mention a scheme keyword → use a lower threshold
            return score >= 0.55
        else:
            # Queries without any keyword → need very high confidence
            return score >= 0.8   # adjust this value as needed


    def select_answer(self, query, retrieved):
        if not retrieved:
            return "Please ask a question related to government schemes."
        top_score = retrieved[0]["score"]
        if not self.is_relevant(query, top_score):
            return "Please ask a question related to government schemes."

        # Multi-answer logic (unchanged)
        MULTI_MARGIN = 0.05
        filtered = [r for r in retrieved if r["score"] >= top_score - MULTI_MARGIN]

        if len(filtered) > 1:
            return " ".join(r["text"] for r in filtered[:2])
        return retrieved[0]["text"]


    def process_text(self, query, user_lang="eng", return_speech=True):
        retrieved = self.retriever.retrieve(query)
        answer_en = self.select_answer(query, retrieved)   # <-- pass query
        answer = self.translator.translate(answer_en, user_lang)
        if return_speech:
            lang = self.lang_map.get(user_lang, "en")
            audio = self.tts.synthesize(answer, lang)
            return answer, audio
        return answer

    def process_speech(self, audio_path, user_lang="eng", return_speech=True):
        query = self.stt.speech_to_text(audio_path)
        print("Recognized text:", query)
        retrieved = self.retriever.retrieve(query)
        answer_en = self.select_answer(query, retrieved)   # <-- pass query here
        answer = self.translator.translate(answer_en, user_lang)
        if return_speech:
            lang = self.lang_map.get(user_lang, "en")
            audio = self.tts.synthesize(answer, lang)
            return answer, audio
        return answer

# YOUR DATASET

schemes_data = [
    {
        "name": "Bharati Scheme for Education",
        "sections": [
            "Bharati Scheme for Education – Description: A financial assistance program for Brahmin students in Andhra Pradesh studying from 1st to 5th class.",
            "Bharati Scheme for Education – Benefits: Eligible students receive ₹5,000 financial assistance per academic year.",
            "Bharati Scheme for Education – Eligibility: Students must belong to the Brahmin community and study in Andhra Pradesh. Family income should not exceed ₹3,00,000 per year.",
            "Bharati Scheme for Education – Priority Categories: Orphans, differently-abled students, and children of single parents are given priority.",
            "Bharati Scheme for Education – Documents Required: Aadhaar card of student and parents, caste certificate, birth certificate, study certificate, and bank passbook.",
            "Bharati Scheme for Education – Where to Apply: Applications must be submitted through the Andhra Pradesh Brahmin Welfare Corporation official portal.",
            "Bharati Scheme for Education – Application Process: Register on the scheme portal, fill the application form with personal details, upload documents, and submit the application.",
            "Bharati Scheme for Education – Selection Authority: Beneficiaries are selected by the State Level Selection Committee of the AP Brahmin Welfare Corporation.",
            "Bharati Scheme for Education – Application Status: Applicants can check status online using Aadhaar number, reference number, or mobile number.",
            "Bharati Scheme for Education – Acknowledgement Slip: Applicants can download the acknowledgement slip from the scheme website after submission."
        ]
    },
    {
        "name": "PM-Kisan Samman Nidhi",
        "sections": [
            "PM-Kisan Samman Nidhi – Description: A central government scheme providing financial support to small and marginal farmers.",
            "PM-Kisan Samman Nidhi – Benefits: Eligible farmers receive ₹6,000 per year in three equal installments directly into their bank accounts.",
            "PM-Kisan Samman Nidhi – Eligibility: Small and marginal farmers owning cultivable land are eligible.",
            "PM-Kisan Samman Nidhi – Documents Required: Aadhaar card, bank account details, and land ownership records.",
            "PM-Kisan Samman Nidhi – Where to Apply: Farmers can apply online through the PM-Kisan portal or via local agriculture offices.",
            "PM-Kisan Samman Nidhi – Payment Method: Payments are transferred directly to beneficiaries through Direct Benefit Transfer (DBT).",
            "PM-Kisan Samman Nidhi – Status Check: Farmers can check payment status using Aadhaar number or registration ID on the official portal."
        ]
    },
    {
        "name": "Ayushman Bharat",
        "sections": [
            "Ayushman Bharat – Description: A national health protection scheme providing health insurance coverage for economically vulnerable families.",
            "Ayushman Bharat – Benefits: Health insurance coverage up to ₹5,00,000 per family per year.",
            "Ayushman Bharat – Eligibility: Families identified in the SECC database are eligible.",
            "Ayushman Bharat – Documents Required: Aadhaar card, ration card, and family identification proof.",
            "Ayushman Bharat – Where to Use: Beneficiaries can receive treatment at empanelled public or private hospitals.",
            "Ayushman Bharat – Registration: Eligible families can register through empanelled hospitals or the official website.",
            "Ayushman Bharat – Services Covered: The scheme covers hospitalization, surgery, medicines, and diagnostic services."
        ]
    },
    {
        "name": "Rythu Bima Scheme",
        "sections": [
            "Rythu Bima Scheme – Description: A Telangana government life insurance scheme for farmers.",
            "Rythu Bima Scheme – Benefits: Insurance coverage of ₹5,00,000 is provided to the nominee in case of the farmer's death.",
            "Rythu Bima Scheme – Eligibility: Farmers aged between 18 and 59 years registered with the agriculture department are eligible.",
            "Rythu Bima Scheme – Documents Required: Aadhaar card, land ownership proof, and bank account details.",
            "Rythu Bima Scheme – Implementation: The scheme is implemented through LIC and the Telangana agriculture department.",
            "Rythu Bima Scheme – Claim Process: Nominees submit required documents to receive insurance compensation."
        ]
    }
]


# BUILD DOCUMENTS

documents = []

for scheme in schemes_data:
    documents.extend(scheme["sections"])



# INITIALIZE SYSTEM

stt_model = WhisperSTT()

translator = Translator()

retriever = JinaRetriever()

retriever.build_index(documents)

tts_model = LocalTTS()

lang_map = {
    "eng": "en",
    "hin": "hi",
    "tel": "te",
    "ben": "bn",
    "urd": "ur"
}

pipeline = IndicQAPipeline(
    stt_model,
    retriever,
    translator,
    tts_model,
    lang_map
)

print("System Ready")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Loading Whisper model...


100%|█████████████████████████████████████| 1.42G/1.42G [00:19<00:00, 77.7MiB/s]


Loading translation model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding shape: (30, 768)
System Ready


In [3]:
uploaded = files.upload()

filename = list(uploaded.keys())[0]

audio = AudioSegment.from_file(filename)

audio = audio.set_frame_rate(16000).set_channels(1)

audio.export("input.wav", format="wav")

answer, audio = pipeline.process_speech(
    "input.wav",
    user_lang="tel"   #TO DETECT TELUGU LANGUAGE
)

print(answer)

display(Audio(audio))

Saving telugu_test (1).mp3 to telugu_test (1).mp3
Recognized text: What are the benefits of PM Kisan Samman Nidhi Padakam to the farmers?
PM-Kisan Samman Nidhi  ప్రయోజనాలుః అర్హత కలిగిన రైతులు తమ బ్యాంకు ఖాతాలకు నేరుగా మూడు సమాన రాయితీలలో సంవత్సరానికి ₹6,000 పొందుతారు. PM-Kisan Samman Nidhi  వివరణః చిన్న మరియు అపరిచిత రైతులకు ఆర్థిక మద్దతును అందించే కేంద్ర ప్రభుత్వ పథకం.


In [8]:
answer, audio = pipeline.process_text(
        "who can apply rythu bima and its benefits?",
        user_lang="eng",
        return_speech=True
    )

print("Answer:", answer)

Answer: Rythu Bima Scheme  Benefits: Insurance coverage of ₹5,00,000 is provided to the nominee in case of the farmer's death.  Eligibility: Farmers aged between 18 and 59 years registered with the department of agriculture are eligible.


In [5]:
questions = [
    ("hin", "आयुष्मान भारत योजना में कितना बीमा कवर मिलता है?"),
    ("tel", "భారతి పథకం కోసం ఎక్కడ దరఖాస్తు చేసుకోవాలి?"),
    ("urd", "پردھان منتری کسان سمن ندھی اسکیم کے فوائد کیا ہیں؟"),
    ("ben", "আয়ুষ্মান ভারত প্রকল্পের জন্য কারা যোগ্য?"),
    ("eng", "What documents are required for Rythu Bima?")
]

for lang, q in questions:

    print(f"\n[{lang}] Question: {q}")

    answer, audio = pipeline.process_text(
        q,
        user_lang=lang,
        return_speech=True
    )

    print("Answer:", answer)

    display(Audio(audio))


[hin] Question: आयुष्मान भारत योजना में कितना बीमा कवर मिलता है?
Answer: आयुष्मान भारत  लाभः स्वास्थ्य बीमा कवरेज प्रति परिवार प्रति वर्ष ₹5,00,000 तक।



[tel] Question: భారతి పథకం కోసం ఎక్కడ దరఖాస్తు చేసుకోవాలి?
Answer: భారతీ స్కీమ్ ఫర్ ఎడ్యుకేషన్  ఎక్కడ దరఖాస్తు చేసుకోవాలిః దరఖాస్తులు ఆంధ్రప్రదేశ్ బ్రాహ్మణ సంక్షేమ సంస్థ అధికారిక పోర్టల్ ద్వారా సమర్పించాలి.



[urd] Question: پردھان منتری کسان سمن ندھی اسکیم کے فوائد کیا ہیں؟
Answer: پی ایم کسان سممان نیازی  فوائد: اہل کسانوں کو سالانہ ₹6،000 تین برابر قسطوں میں براہ راست اپنے بینک اکاؤنٹس میں ملتے ہیں۔ ریتھو بیما اسکیم  فوائد: کسان کی موت کی صورت میں نامزد امیدوار کو ₹5،000 کی انشورنس کوریج فراہم کی جاتی ہے۔



[ben] Question: আয়ুষ্মান ভারত প্রকল্পের জন্য কারা যোগ্য?
Answer: সরকারি প্রকল্পের সাথে সম্পর্কিত একটি প্রশ্ন করুন।



[eng] Question: What documents are required for Rythu Bima?
Answer: Rythu Bima Scheme  Documents Required: Aadhaar card, proof of land ownership, and bank account details.  Claim Process: Nominees submit the required documents to receive insurance compensation.


In [9]:
answer, audio = pipeline.process_text(
        "what is ai?",
        user_lang="eng",
        return_speech=True
    )

print("Answer:", answer)

Answer: Please ask a question related to government schemes.
